# Pinecone을 활용한 검색 증강 생성(RAG)

이 노트북에서는 검색 증강 생성(RAG, retrieval-augmented generation)이라는 기법으로 Claude를 Pinecone 벡터 데이터베이스의 데이터와 연결하는 방법을 보여 줍니다. 다음 단계를 다룹니다.

1. Voyage AI의 임베딩 모델로 데이터셋 임베딩하기
2. 임베딩을 Pinecone 인덱스에 업로드하기
3. 벡터 데이터베이스에서 정보 검색하기
4. 데이터베이스의 정보를 활용해 Claude로 질문에 답하기

## 준비
먼저 이 노트북에서 사용할 라이브러리를 설치하고 필요한 API 키를 설정합니다. [Claude API 키](https://docs.claude.com/claude/reference/getting-started-with-the-api), 무료 [Pinecone API 키](https://docs.pinecone.io/docs/quickstart), 무료 [Voyage AI API 키](https://docs.voyageai.com/install/)가 필요합니다.

In [ ]:
%pip install anthropic datasets pinecone-client voyageai

In [ ]:
# Insert your API keys here
ANTHROPIC_API_KEY = "<YOUR_ANTHROPIC_API_KEY>"
PINECONE_API_KEY = "<YOUR_PINECONE_API_KEY>"
VOYAGE_API_KEY = "<YOUR_VOYAGE_API_KEY>"

## 데이터셋 내려받기
1만 개가 넘는 아마존 상품 설명이 담긴 Amazon products 데이터셋을 내려받아 DataFrame으로 불러오겠습니다.

In [ ]:
import pandas as pd

# Download the JSONL file
!wget  https://www-cdn.anthropic.com/48affa556a5af1de657d426bcc1506cdf7e2f68e/amazon-products.jsonl

data = []
with open("amazon-products.jsonl") as file:
    for line in file:
        try:
            data.append(eval(line))  # noqa: S307
        except (SyntaxError, ValueError):
            # Skip malformed lines in the dataset
            pass

df = pd.DataFrame(data)
display(df.head())
len(df)

## 벡터 데이터베이스

벡터 데이터베이스를 만들려면 먼저 Pinecone의 무료 API 키가 필요합니다. 키를 확보하면 다음과 같이 데이터베이스를 초기화할 수 있습니다:

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

다음으로 인덱스 사양을 설정합니다. 인덱스를 배포할 클라우드 제공자와 리전을 지정할 수 있습니다. 사용 가능한 제공자와 리전 목록은 [여기](https://www.pinecone.io/docs/data-types/metadata/)에서 확인할 수 있습니다.

In [ ]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(cloud="aws", region="us-west-2")

그런 다음 인덱스를 초기화합니다. 임베딩 생성에는 Voyage의 "voyage-2" 모델을 사용하므로 차원을 1024로 설정합니다.

In [ ]:
import time

index_name = "amazon-products"
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1024,  # dimensionality of voyage-2 embeddings
        metric="dotproduct",
        spec=spec,
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

아직 벡터를 추가하지 않았으므로 새 Pinecone 인덱스의 total_vector_count가 0으로 나타나야 합니다.

## 임베딩
Voyage 임베딩을 시작하려면 [여기](https://www.voyageai.com)에서 API 키를 발급받으세요.

이제 Voyage 클라이언트를 설정하고 `embed` 메서드로 임베딩을 만드는 방법을 살펴보겠습니다. Claude와 함께 Voyage 임베딩을 사용하는 방법을 더 알아보려면 [이 노트북](https://github.com/anthropics/anthropic-cookbook/blob/main/third_party/VoyageAI/how_to_create_embeddings.md)을 참고하세요.

In [ ]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

texts = ["Sample text 1", "Sample text 2"]

result = vo.embed(texts, model="voyage-2", input_type="document")
print(result.embeddings[0])
print(result.embeddings[1])

## Pinecone 인덱스에 데이터 업로드하기

임베딩 모델을 설정했으니, 이제 상품 설명을 임베딩해 Pinecone 인덱스에 업로드할 수 있습니다.

In [ ]:
from time import sleep

from tqdm.auto import tqdm

descriptions = df["text"].tolist()
batch_size = 100  # how many embeddings we create and insert at once

for i in tqdm(range(0, len(descriptions), batch_size)):
    # find end of batch
    i_end = min(len(descriptions), i + batch_size)
    descriptions_batch = descriptions[i:i_end]
    # create embeddings (try-except added to avoid RateLimitError. Voyage currently allows 300/requests per minute.)
    done = False
    while not done:
        try:
            res = vo.embed(descriptions_batch, model="voyage-2", input_type="document")
            done = True
        except Exception:
            sleep(5)

    embeds = [record for record in res.embeddings]
    # create unique IDs for each text
    ids_batch = [f"description_{idx}" for idx in range(i, i_end)]

    # Create metadata dictionaries for each text
    metadata_batch = [{"description": description} for description in descriptions_batch]

    to_upsert = list(zip(ids_batch, embeds, metadata_batch, strict=False))

    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

## 질의하기

인덱스를 채웠으니 질의해 결과를 얻을 수 있습니다. 자연어 질문을 임베딩한 뒤 인덱스에 질의해 의미적으로 유사한 상품 설명을 반환받을 수 있습니다.

In [49]:
USER_QUESTION = (
    "I want to get my daughter more interested in science. What kind of gifts should I get her?"
)

question_embed = vo.embed([USER_QUESTION], model="voyage-2", input_type="query")
results = index.query(vector=question_embed.embeddings, top_k=5, include_metadata=True)
results

{'matches': [{'id': 'description_1771',
              'metadata': {'description': 'Product Name: Scientific Explorer '
                                          'My First Science Kids Science '
                                          'Experiment Kit\n'
                                          '\n'
                                          'About Product: Experiments to spark '
                                          'creativity and curiosity | Grow '
                                          'watery crystals, create a rainbow '
                                          'in a plate, explore the science of '
                                          'color and more | Represents STEM '
                                          '(Science, Technology, Engineering, '
                                          'Math) principles – open ended toys '
                                          'to construct, engineer, explorer '
                                          'and experiment | Inclu

## 검색 최적화하기

결과가 괜찮지만 더 개선할 수 있습니다. Claude를 활용해 사용자의 질문에서 검색 키워드를 생성할 수 있습니다. 이렇게 하면 인덱스를 폭넓고 다양하게 검색해 더 관련성 높은 상품 설명을 얻을 수 있습니다.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def get_completion(prompt):
    completion = client.completions.create(
        model="claude-2.1",
        prompt=prompt,
        max_tokens_to_sample=1024,
    )
    return completion.completion

In [ ]:
def create_keyword_prompt(question):
    return f"""\n\nHuman: Given a question, generate a list of 5 very diverse search keywords that can be used to search for products on Amazon.

The question is: {question}

Output your keywords as a JSON that has one property "keywords" that is a list of strings. Only output valid JSON.\n\nAssistant:{{"""

Anthropic 클라이언트를 설정하고 프롬프트를 만들었으니, 이제 질문에서 키워드를 생성해 보겠습니다. Claude의 출력에서 쉽게 파싱할 수 있도록 키워드를 JSON 객체로 출력하게 합니다.

In [ ]:
keyword_json = "{" + get_completion(create_keyword_prompt(USER_QUESTION))
print(keyword_json)

In [ ]:
import json

# Extract the keywords from the JSON
data = json.loads(keyword_json)
keywords_list = data["keywords"]
print(keywords_list)

이제 키워드 목록이 생겼으니, 각각을 임베딩해 인덱스에 질의하고 가장 관련성 높은 상품 설명 3개를 반환받겠습니다.

In [ ]:
results_list = []
for keyword in keywords_list:
    # get the embeddings for the keywords
    query_embed = vo.embed([keyword], model="voyage-2", input_type="query")
    # search for the embeddings in the Pinecone index
    search_results = index.query(vector=query_embed.embeddings, top_k=3, include_metadata=True)
    # append the search results to the list
    for search_result in search_results.matches:
        results_list.append(search_result["metadata"]["description"])
print(len(results_list))

## Claude로 답변하기

상품 설명 목록이 준비되었으니, Claude가 학습한 검색 템플릿 형식으로 정리해 다른 프롬프트에 넣어 보겠습니다.

In [47]:
# Formatting search results
def format_results(extracted: list[str]) -> str:
    result = "\n".join(
        [
            f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
            for i, r in enumerate(extracted)
        ]
    )

    return f"\n<search_results>\n{result}\n</search_results>"


def create_answer_prompt(results_list, question):
    return f"""\n\nHuman: {format_results(results_list)} Using the search results provided within the <search_results></search_results> tags, please answer the following question <question>{question}</question>. Do not reference the search results in your answer.\n\nAssistant:"""

마지막으로 원래의 사용자 질문을 던져 Claude에게서 답을 받아 보겠습니다.

In [50]:
answer = get_completion(create_answer_prompt(results_list, USER_QUESTION))
print(answer)

 To get your daughter more interested in science, I would recommend getting her an age-appropriate science kit or set that allows for hands-on exploration and experimentation. For example, for a younger child you could try a beginner chemistry set, magnet set, or crystal growing kit. For an older child, look for kits that tackle more advanced scientific principles like physics, engineering, robotics, etc. The key is choosing something that sparks her natural curiosity and lets her actively investigate concepts through activities, observations, and discovery. Supplement the kits with science books, museum visits, documentaries, and conversations about science she encounters in everyday life. Making science fun and engaging is crucial for building her interest.
